# Avaliação offline e casos explicáveis

A simulação atribui recompensas contrafactuais previstas pelo modelo de recompensa ajustado no treino; por isso, seu resultado depende da qualidade desse modelo e não é evidência causal. A avaliação off-policy usa ações e recompensas históricas, registra overlap e tamanho efetivo da amostra para expor limitações de suporte. Os cinco casos explicáveis são retirados do teste e mostram recomendações para contextos reais da base, sem representar resultados contrafactuais observados.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
PROJECT_ROOT = next(path for path in candidates if (path / 'src' / 'adaptive_offers').exists())
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import json

from adaptive_offers.config import DEFAULT_DATA_PATH
from adaptive_offers.data import load_bank_marketing, validate_schema
from adaptive_offers.evaluation import evaluate_policy_suite
from adaptive_offers.golden_set import build_golden_set
from adaptive_offers.policies import Guardrails, ThompsonSamplingPolicy
from adaptive_offers.preparation import modeling_frame, temporal_split
from adaptive_offers.reward_model import ChannelRewardModel

df = load_bank_marketing(DEFAULT_DATA_PATH)
validate_schema(df)
splits = temporal_split(modeling_frame(df))
model = ChannelRewardModel.fit(splits.train)
report_candidates = sorted((PROJECT_ROOT / 'artifacts').glob('*evaluation_report.json'))
report_path = report_candidates[0] if report_candidates else None
if report_path is not None:
    report = json.loads(report_path.read_text())
    simulation = pd.DataFrame(report['simulation'])
    ope = pd.DataFrame(report['ope'])
    print('Tabela oficial carregada do relatório de avaliação local.')
    print('Desacordo entre o modelo contextual e o baseline fixo:', report.get('context_disagreement_rate'))
else:
    simulation, ope = evaluate_policy_suite(
        splits.train, splits.test, model, seeds=tuple(range(5)),
    )
    print('Artefato oficial ausente. Comparação executada com 5 seeds. Use make evaluate para 30.')
display(simulation)
display(ope[['policy', 'ips', 'snips', 'doubly_robust', 'effective_sample_size', 'clip_rate']])

In [ ]:
baseline = simulation.set_index('policy').loc['fixed_best']
adaptive = simulation.set_index('policy').loc['thompson_sampling']
print(
    f"Baseline fixo: {baseline['conversion_rate']:.4f} | "
    f"Thompson contextual: {adaptive['conversion_rate']:.4f} | "
    f"uplift absoluto: {adaptive['uplift_absolute']:.4f} "
    f"(IC {adaptive['uplift_ci_low']:.4f} a {adaptive['uplift_ci_high']:.4f})"
)
ope_indexed = ope.set_index('policy')
print(
    'SNIPS baseline:', round(float(ope_indexed.loc['fixed_best', 'snips']), 4),
    '| SNIPS Thompson:', round(float(ope_indexed.loc['thompson_sampling', 'snips']), 4),
    '| clip rate:', round(float(ope_indexed.loc['thompson_sampling', 'clip_rate']), 4),
)
simulation_gain = float(adaptive['uplift_ci_low']) > 0
ope_stable = float(ope_indexed.loc['thompson_sampling', 'clip_rate']) < 0.1
if simulation_gain and ope_stable:
    print('Uplift da simulação com OPE estável o suficiente para discutir ganho.')
elif simulation_gain:
    print(
        'A simulação semissintética tem IC de uplift acima de zero. '
        'O OPE não confirma: clipping alto ou estimadores fora da faixa de uma taxa. '
        'Não tratar o número simulado como efeito causal do canal.'
    )
else:
    print('O intervalo do uplift não fica acima de zero. O resultado permanece na tabela, sem escolher seed.')

## Golden Set de cinco casos

In [ ]:
policy = ThompsonSamplingPolicy(
    model.actions, seed=42, guardrails=Guardrails(), reward_model=model,
).fit(splits.train)
golden = build_golden_set(splits.test, model, policy)
golden_table = pd.DataFrame([{
    'case_id': item['case_id'],
    'recommended_action': item['recommendation']['action'],
    'action_probability': item['recommendation']['action_probability'],
    'reason': item['recommendation']['reason'],
} for item in golden])
display(golden_table)